In [1]:
import torch 
import torch.nn as nn
import math 

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, d_ff, dropout=0.1, max_length=50_000):
        super().__init__()
        self.d_model = d_model
        self.vocab_emb = nn.Embedding(vocab_size, d_model)
        # a simple linear pos encoding
        # self.pos_encoding = nn.Embedding(max_length, d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=n_head,
            num_encoder_layers=n_layer,
            num_decoder_layers=n_layer,
            dim_feedforward=d_ff,
            dropout=dropout
        )
        self.ow = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('pos_encoding', self.create_sin_cos_position_embedding(max_length, d_model))
        
    def create_sin_cos_position_embedding(self, max_length, d_model):
        position = torch.arange(0, max_length).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10_000.0) / d_model))
        pe = torch.zeros(max_length, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(1)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        src_seq_len = src.size(-2)
        tgt_seq_len = tgt.size(-2)
        src = self.dropout(self.vocab_emb(src) + self.pos_encoding[:src_seq_len, :])
        tgt = self.dropout(self.vocab_emb(tgt) + self.pos_encoding[:tgt_seq_len, :])
        out = self.transformer(src, tgt, src_mask=src_mask, tgt_mask=tgt_mask)
        return self.ow(out)


In [2]:

# Example usage
vocab_size = 10000
embed_size = 512
num_heads = 8
num_encoder_layers = 6
num_decoder_layers = 6
forward_expansion = 2048
dropout = 0.1
max_length = 100
    
model = TransformerModel(
    vocab_size,
    embed_size,
    num_heads,
    num_encoder_layers,
    forward_expansion,
    dropout,
    max_length
)

# Dummy input
src = torch.randint(0, vocab_size, (50, 32))  # (source sequence length, batch size)
trg = torch.randint(0, vocab_size, (50, 32))  # (target sequence length, batch size)

output = model(src, trg)
print(output.shape)  # Expected shape: (target sequence length, batch size, vocab size)


/home/kennethwang/miniconda3/envs/py-notebook/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


torch.Size([50, 32, 10000])


In [3]:
# Load TED talks dataset for Portuguese-English translation
import tensorflow_datasets as tfds
import tensorflow as tf
import torch.optim as optim
import time

# Load the dataset
train_examples, val_examples, test_examples = tfds.load(
    'ted_hrlr_translate/pt_to_en',
    split=['train', 'validation', 'test'],
    as_supervised=True)

# Create tokenizers
tokenizers = {}
for lang, data in [('pt', [ex[0].numpy().decode('utf-8') for ex in train_examples]), 
                   ('en', [ex[1].numpy().decode('utf-8') for ex in train_examples])]:
    tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
        data, target_vocab_size=2**12)  # Reduced vocab size
    tokenizers[lang] = tokenizer

# Constants for special tokens
START_TOKEN = [tokenizers['pt'].vocab_size]
END_TOKEN = [tokenizers['pt'].vocab_size + 1]
VOCAB_SIZE_PT = tokenizers['pt'].vocab_size + 2
VOCAB_SIZE_EN = tokenizers['en'].vocab_size + 2

# Preprocessing function
def preprocess_text(pt, en):
    pt = START_TOKEN + tokenizers['pt'].encode(pt.numpy().decode('utf-8')) + END_TOKEN
    en = START_TOKEN + tokenizers['en'].encode(en.numpy().decode('utf-8')) + END_TOKEN
    return pt, en

def tf_preprocess(pt, en):
    return tf.py_function(preprocess_text, [pt, en], [tf.int64, tf.int64])

# Create TF datasets with reduced sizes
BUFFER_SIZE = 1000  # Reduced buffer size
BATCH_SIZE = 4  # Reduced batch size
MAX_LENGTH = 20  # Reduced max sequence length

def filter_max_length(pt, en):
    return tf.logical_and(tf.size(pt) <= MAX_LENGTH,
                         tf.size(en) <= MAX_LENGTH)

train_dataset = (train_examples
                .map(tf_preprocess)
                .filter(filter_max_length)
                .shuffle(BUFFER_SIZE)
                .padded_batch(BATCH_SIZE, padded_shapes=([None], [None]))
                .prefetch(tf.data.AUTOTUNE))

val_dataset = (val_examples
              .map(tf_preprocess)
              .filter(filter_max_length)
              .padded_batch(BATCH_SIZE, padded_shapes=([None], [None])))

# Convert TF dataset to PyTorch
def tf_to_torch(tf_dataset):
    for pt_batch, en_batch in tf_dataset:
        pt_tensor = torch.LongTensor(pt_batch.numpy())
        en_tensor = torch.LongTensor(en_batch.numpy())
        yield pt_tensor.T, en_tensor.T  # Transpose to match PyTorch expected shape

# Training function
def train_epoch(model, optimizer, criterion, train_data, device):
    model.train()
    losses = 0
    for src, tgt in tf_to_torch(train_data):
        src = src.to(device)
        tgt = tgt.to(device)
        
        # Create masks
        src_mask = None  # For encoder self-attention
        tgt_len = tgt.shape[0] - 1  # Adjust for teacher forcing
        # Create square subsequent mask for decoder self-attention
        tgt_mask = torch.triu(torch.ones(tgt_len, tgt_len) * float('-inf'), diagonal=1).to(device)
        
        optimizer.zero_grad()
        output = model(src, tgt[:-1], src_mask, tgt_mask)
        
        # Ensure target values are within valid range
        tgt_labels = tgt[1:].reshape(-1)
        tgt_labels = torch.clamp(tgt_labels, 0, VOCAB_SIZE_EN - 1)
        
        loss = criterion(output.reshape(-1, output.shape[-1]), tgt_labels)
        loss.backward()
        optimizer.step()
        
        losses += loss.item()
        print(f"loss{loss.item()}")
    
    return losses / len(list(train_data))

# Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Update model parameters for new vocabulary sizes
model.vocab_emb = nn.Embedding(VOCAB_SIZE_PT, embed_size)
model.ow = nn.Linear(embed_size, VOCAB_SIZE_EN)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Using 0 as padding index

# Training loop
NUM_EPOCHS = 1
for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    train_loss = train_epoch(model, optimizer, criterion, train_dataset, device)
    end_time = time.time()
    
    print(f"Epoch: {epoch+1}, Train loss: {train_loss:.3f}, Epoch time: {(end_time - start_time):.2f}s")


2025-03-30 19:05:15.551919: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743386715.566732   36720 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743386715.571036   36720 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-30 19:05:15.592334: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1743386718.440298   36720 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 90

loss9.058772087097168
loss8.745671272277832
loss8.651249885559082
loss8.335039138793945
loss8.33846664428711
loss8.133905410766602
loss8.058399200439453
loss7.905728340148926
loss7.843776226043701
loss7.887817859649658
loss7.638893127441406
loss7.605636119842529
loss7.820685863494873
loss7.562377452850342
loss7.499462127685547
loss7.608072757720947
loss7.70105504989624
loss7.389822006225586
loss7.5325469970703125
loss7.255326271057129
loss7.359367847442627
loss7.46832275390625
loss7.360466480255127
loss7.306220531463623
loss7.401390075683594
loss7.812559604644775
loss7.571852684020996
loss7.17224645614624
loss7.099849224090576
loss7.3565754890441895
loss7.240653038024902
loss6.892292499542236
loss7.217496395111084
loss7.151390075683594
loss7.363086700439453
loss7.246113300323486
loss6.90256929397583
loss7.204697608947754
loss6.875144958496094
loss7.005851745605469
loss6.912620544433594
loss6.773599147796631
loss7.015369415283203
loss7.214597225189209
loss6.928659439086914
loss6.7832088

KeyboardInterrupt: 